### Aula 08 — Parquet e leitura seletiva

O objetivo é relacionar a organização dos dados no armazenamento com o trabalho necessário para consultar pedidos. O percurso real deste notebook é: ler CSV, escrever e reler Parquet, selecionar colunas, enviar filtros ao leitor e comparar um dataset comum com outro particionado por `status`.

**Pré-requisitos:** conceitos de DataFrame, transformações e actions; ambiente PySpark com acesso a `/workspace/datasets/08_parquet/pedidos.csv` e permissão de escrita na mesma pasta. As escritas usam `overwrite`, portanto substituem os destinos indicados. Os resultados abaixo pertencem à execução já salva; os planos mostram decisões de leitura, sem constituir um benchmark.

#### 1. Preparar a sessão

`SparkSession` fornece a entrada para leitura e escrita distribuída; `F` reúne funções que produzem expressões de colunas. `getOrCreate()` cria ou reutiliza uma sessão. As células seguintes mantêm o nome de aplicação usado no laboratório.


In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_08_parquet")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 18:33:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### 2. Ler o CSV e conferir o schema

`header=True` usa a primeira linha como nomes das colunas; `inferSchema=True` solicita inferência de tipos a partir dos dados, o que pode exigir leitura adicional. CSV guarda texto delimitado; em uma ingestão controlada também seria possível fornecer um schema explícito.

**Observado:** `printSchema()` apresenta seis colunas: IDs inteiros, `data_pedido` como `date`, `status`/`canal` como strings e `valor_total` como `double`. O `show(5)` executa uma consulta para exibir uma amostra; cinco linhas não validam a integridade nem a contagem total do dataset. Definir o DataFrame e inspecionar seu schema não equivale a materializar todas as linhas para processamento.


In [3]:
df_csv = spark.read.csv("/workspace/datasets/08_parquet/pedidos.csv", header= True, inferSchema= True)

In [4]:
df_csv.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [5]:
df_csv.show(5)

+---------+----------+-----------+---------+-----------+-----------+
|pedido_id|cliente_id|data_pedido|   status|      canal|valor_total|
+---------+----------+-----------+---------+-----------+-----------+
|    10001|       452| 2025-10-09|concluido|       site|     473.56|
|    10002|        71| 2025-08-03|concluido|        app|     303.13|
|    10003|       411| 2025-10-06|  enviado|       loja|     281.64|
|    10004|       184| 2025-09-03|concluido|       loja|     281.65|
|    10005|       290| 2025-09-15|concluido|marketplace|     893.22|
+---------+----------+-----------+---------+-----------+-----------+
only showing top 5 rows



#### 3. Converter para Parquet e recuperar os tipos

A escrita é uma action: executa o necessário para gravar `df_csv` no destino. O caminho identifica uma **pasta de dataset**, que pode conter vários arquivos `part-*` e metadados. A releitura deve considerar essa pasta, e não um único arquivo isolado.

Parquet armazena dados tipados em formato colunar. Por isso, `spark.read.parquet()` recupera o schema sem `inferSchema`; o schema salvo a seguir confirma os mesmos seis campos e tipos do CSV inferido. Não há comparação de tamanhos ou duração nesta sequência.

**Compressão:** este notebook não fixa um codec. O [desafio 08](../desafios/desafio_08_parquet.ipynb) explicita Snappy na escrita; isso não deve ser atribuído como configuração explícita às células desta aula. Compressão pode reduzir armazenamento e I/O, mas seu efeito no tempo depende também do custo de CPU e da consulta.


In [6]:
df_csv.write.mode("overwrite").parquet("/workspace/datasets/08_parquet/arquivos_parquet")

In [7]:
df_parquet = spark.read.parquet("/workspace/datasets/08_parquet/arquivos_parquet")

In [8]:
df_parquet.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



#### 4. Column Pruning: ler somente as colunas necessárias

`select("cliente_id", "valor_total")` cria uma projeção lazy. `explain(True)` mostra os planos parsed, analyzed, optimized e physical: o Catalyst resolve nomes/tipos e planeja a execução com as colunas necessárias.

**Observado no Physical Plan:** `FileScan parquet` contém `ReadSchema: struct<cliente_id:int,valor_total:double>`, em vez dos seis campos da origem. Essa é a evidência de Column Pruning no planejamento da leitura. `Batched: true` indica leitura em lotes colunares; `ColumnarToRow` adapta a saída para operadores que trabalham com linhas. `InMemoryFileIndex` é o índice de arquivos da fonte, não o cache do DataFrame.

O armazenamento colunar permite evitar colunas de dados não utilizadas. O plano, isoladamente, não informa bytes efetivamente lidos nem um ganho de desempenho medido.


In [9]:
df_parquet_select = (
df_parquet.select("cliente_id", "valor_total")
)

In [10]:
df_parquet_select.explain(True)

== Parsed Logical Plan ==
'Project ['cliente_id, 'valor_total]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Project [cliente_id#74, valor_total#78]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Optimized Logical Plan ==
Project [cliente_id#74, valor_total#78]
+- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [cliente_id#74,valor_total#78] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/arquivos_parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,valor_total:double>



#### 5. Predicate Pushdown: aproximar o filtro do leitor

O filtro mantém somente pedidos com `valor_total > 500`, depois da seleção das duas colunas. A transformação continua lazy; `explain(True)` descreve o plano sem executar uma consulta completa de linhas.

**Observado:** `ReadSchema` continua reduzido; `DataFilters` registra `isnotnull(valor_total)` e `valor_total > 500.0`; `PushedFilters` contém `IsNotNull` e `GreaterThan`. O filtro foi encaminhado ao leitor Parquet. O `Filter` também permanece acima de `FileScan`, garantindo a condição sobre as linhas retornadas pelo leitor. O teste de nulo é compatível com a comparação: um valor nulo não torna `> 500` verdadeiro.

Pushdown pode permitir descartar blocos incompatíveis por metadados. A presença de `PushedFilters` **não prova** que algum Row Group foi efetivamente descartado nesta execução; isso exigiria evidências adicionais. Não há contagem de pedidos filtrados salva neste bloco.


In [11]:
df_pushdown = (
    df_parquet.select("cliente_id", "valor_total")
    .filter(F.col("valor_total") > 500)
)

In [12]:
df_pushdown.explain(True)

== Parsed Logical Plan ==
'Filter ('valor_total > 500)
+- Project [cliente_id#74, valor_total#78]
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Analyzed Logical Plan ==
cliente_id: int, valor_total: double
Filter (valor_total#78 > cast(500 as double))
+- Project [cliente_id#74, valor_total#78]
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Optimized Logical Plan ==
Project [cliente_id#74, valor_total#78]
+- Filter (isnotnull(valor_total#78) AND (valor_total#78 > 500.0))
   +- Relation [pedido_id#73,cliente_id#74,data_pedido#75,status#76,canal#77,valor_total#78] parquet

== Physical Plan ==
*(1) Filter (isnotnull(valor_total#78) AND (valor_total#78 > 500.0))
+- *(1) ColumnarToRow
   +- FileScan parquet [cliente_id#74,valor_total#78] Batched: true, DataFilters: [isnotnull(valor_total#78), (valor_total#78 > 500.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file

#### 6. Preparar duas organizações físicas do mesmo conjunto

A primeira escrita cria `pedidos_normal`, sem `partitionBy`, e sua releitura fica disponível em `df_parquet_pedidos`. Em seguida, o notebook grava a mesma origem por `status`. A sequência prepara um contraste de organização no storage; não contém uma medição comparativa de velocidade ou volume lido entre os dois destinos.


In [13]:
df_csv.write.mode("overwrite").parquet("/workspace/datasets/08_parquet/pedidos_normal")

In [14]:
df_parquet_pedidos = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_normal")

##### 6.1. `partitionBy("status")`: diretórios por valor

A escrita organiza os arquivos em diretórios como `status=concluido/`. Uma partição física pode conter vários arquivos: ela não corresponde obrigatoriamente a um arquivo nem a uma Spark Partition.

Ao reler a pasta raiz, Spark descobre os valores de `status` pelos diretórios. **Observado no plano sem filtro:** a relação oferece os seis campos, mas o `ReadSchema` dos arquivos tem cinco e omite `status`. A coluna existe no DataFrame porque é reconstruída a partir do caminho. `PartitionFilters: []` indica que esta consulta ainda não restringiu os valores de partição.


In [15]:
df_csv.write.mode("overwrite").partitionBy("status").parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [16]:
df_pedidos_status = spark.read.parquet("/workspace/datasets/08_parquet/pedidos_por_status")

In [17]:
df_pedidos_status.explain(True)

== Parsed Logical Plan ==
Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Optimized Logical Plan ==
Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<pedido_id:int,cliente_id:int,data_pedido:date,canal:string,valor_total:double>



#### 7. Partition Pruning: selecionar os diretórios relevantes

O filtro `status == "concluido"` atua sobre a coluna de particionamento físico. **Observado:** o plano passa a exibir `PartitionFilters: [isnotnull(status), (status = concluido)]`, enquanto `PushedFilters` e `DataFilters` ficam vazios.

A condição de partição permite excluir os arquivos dos demais valores de `status` da leitura de dados. A descoberta de partições ainda pode consultar diretórios e metadados. Aqui `PushedFilters: []` não representa falha: o filtro está na organização do dataset, não em uma coluna armazenada dentro de cada arquivo. Não há métrica salva de arquivos ou bytes evitados.


In [18]:
df_status_pruning = (
    df_pedidos_status.filter(F.col("status") == "concluido")
)

In [19]:
df_status_pruning.explain(True)

== Parsed Logical Plan ==
'Filter ('status = concluido)
+- Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Analyzed Logical Plan ==
pedido_id: int, cliente_id: int, data_pedido: date, canal: string, valor_total: double, status: string
Filter (status#132 = concluido)
+- Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Optimized Logical Plan ==
Filter (isnotnull(status#132) AND (status#132 = concluido))
+- Relation [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [pedido_id#127,cliente_id#128,data_pedido#129,canal#130,valor_total#131,status#132] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/08_parquet/pedidos_por_status], PartitionFilters: [isnotnull(status#132), (status#132 = concluido)], PushedFilters: [], Rea

#### 8. Row Groups, partições e o risco de arquivos pequenos

As três estruturas atuam em níveis diferentes:

| Estrutura | Papel neste contexto |
| --- | --- |
| Spark Partition | Unidade de distribuição do trabalho entre tasks; `repartition()` altera essa distribuição. |
| Partição física por `status` | Diretórios do dataset produzidos por `write.partitionBy()`. |
| Row Group | Bloco interno do Parquet, com partes colunares e possíveis estatísticas, como mínimo/máximo. |

Para `valor_total > 500`, um Row Group cujo máximo seja 300 poderia ser descartado se a estatística estiver disponível e for utilizável; um máximo 900 apenas exige considerar o bloco. **Esse exemplo é conceitual**, não uma inspeção dos arquivos desta aula. Data skipping é o descarte efetivo de blocos; pushdown é a entrega da condição ao leitor.

A escrita por `cliente_id` abaixo está comentada. Alta cardinalidade pode multiplicar diretórios e arquivos pequenos, aumentando o custo de metadados e agendamento. Não há resultado de execução dessa escrita na célula comentada. Uma boa chave física precisa combinar filtros recorrentes com volume suficiente por partição.


In [20]:
#cardinalidade gigantesca decisão ruim salvar por partição
#df_csv.write.mode("overwrite").partitionBy("cliente_id").parquet("/workspace/datasets/08_parquet/pedidos_por_clientes")

#### 9. Encerrar a sessão e interpretar o registro preservado

`spark.stop()` encerra a sessão e seu contexto. A célula original tem contador de execução ausente e conserva um log `FileFormatWriter: Aborting job`, com `NoSuchElementException: None.get`. A localização desse log não permite concluir que `stop()` foi a causa, nem associá-lo com segurança a uma escrita específica. O registro é mantido como evidência histórica; os arquivos de saída e a conclusão das escritas precisariam ser verificados em uma reprodução, que não faz parte desta documentação.


In [ ]:
spark.stop()

26/09/30 00:55:48 ERROR FileFormatWriter: Aborting job d328cccc-af2c-47f8-abac-e35e33fc9eac.
java.util.NoSuchElementException: None.get
	at scala.None$.get(Option.scala:529)
	at scala.None$.get(Option.scala:527)
	at org.apache.spark.sql.execution.datasources.BasicWriteJobStatsTracker.processStats(BasicWriteStatsTracker.scala:202)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$processStats$5(FileFormatWriter.scala:448)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.$anonfun$processStats$5$adapted(FileFormatWriter.scala:447)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.processStats(FileFormatWriter.scala:447)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.writeAndCommit(FileFormatWriter.scala:278)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeWrite(FileFormatWriter.scala:304)
	at org.apache.spark.sql.execution.datasources

#### Resumo dos aprendizados

- O Parquet recuperado preserva os tipos que aparecem no schema salvo, sem inferência de CSV.
- `ReadSchema` reduzido evidencia Column Pruning; `PushedFilters` mostra condições enviadas ao leitor; `PartitionFilters` registra a seleção de partições físicas.
- Row Groups, Spark Partitions e diretórios de `partitionBy()` são estruturas diferentes. Nenhum dos planos salvos comprova quantos Row Groups foram descartados.
- Particionar por uma chave de alta cardinalidade pode gerar arquivos pequenos; a tentativa por `cliente_id` permanece comentada.
- Os planos explicam mecanismos, sem comprovar aceleração, compressão efetiva ou leitura de bytes. O log final impede tratar todo o histórico como uma execução limpa e linear.

**Para continuar:** [material da aula](../docs/08_parquet.md), [desafio 08](../desafios/desafio_08_parquet.ipynb) e [aula 09 — JDBC](aula_09_jdbc.ipynb).
